# NBA Dataset Exploration
This notebook explores the NBA games dataset to understand its structure and contents.

# Stage 1: Data Understanding and Baseline
Load data, understand structure, create first target variable.

1. Load data

In [231]:
import pandas as pd

df = pd.read_csv("../data/nba_games.csv")
df.head()
df.info()
print(df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 73279 entries, 0 to 73278
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gameId            73279 non-null  int64  
 1   gameDateTimeEst   73279 non-null  str    
 2   hometeamCity      73272 non-null  str    
 3   hometeamName      73279 non-null  str    
 4   hometeamId        73279 non-null  int64  
 5   awayteamCity      73272 non-null  str    
 6   awayteamName      73279 non-null  str    
 7   awayteamId        73279 non-null  int64  
 8   homeScore         73279 non-null  int64  
 9   awayScore         73279 non-null  int64  
 10  winner            73279 non-null  int64  
 11  gameType          73279 non-null  str    
 12  gameSubtype       74 non-null     str    
 13  gameLabel         4043 non-null   str    
 14  gameSubLabel      328 non-null    str    
 15  seriesGameNumber  5823 non-null   object 
 16  attendance        1392 non-null   float64
 17  aren

/var/folders/gd/hhnfp6j538bfgrk8b6_6vxf00000gn/T/ipykernel_19524/4259515833.py:3: DtypeWarning: Columns (0: gameSubtype, 1: gameSubLabel, 2: seriesGameNumber, 3: arenaName, 4: arenaCity, 5: arenaState, 6: officials) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/nba_games.csv")


2. Create target variable

In [232]:
# 1 = home win, 0 = away win
df["HOME_WIN"] = (df["homeScore"] > df["awayScore"]).astype(int)

3. Baseline metric

In [233]:
home_win_rate = df["HOME_WIN"].mean()
print(f"Home Win Rate: {home_win_rate:.2%}")

away_win_rate = 1 - home_win_rate
print(f"Away Win Rate: {away_win_rate:.2%}")

Home Win Rate: 61.57%
Away Win Rate: 38.43%


# Stage 2: Team Strength
Turn raw games into team-level performance metrics.

1. Filter modern era

In [234]:
df_modern = df[df["gameDate"] >= "2000-01-01"].copy()
df_modern = df_modern.sort_values("gameDate")

2. Recreate target

In [235]:
df_modern["HOME_WIN"] = (df_modern["homeScore"] > df_modern["awayScore"]).astype(int)
df_modern["AWAY_WIN"] = (df_modern["awayScore"] > df_modern["homeScore"]).astype(int)

3. Team strength (home + away)

In [236]:
home_strength = df_modern.groupby("hometeamName")["HOME_WIN"].mean()
away_strength = df_modern.groupby("awayteamName")["AWAY_WIN"].mean()

4. Combine into one table

In [237]:
team_strength = pd.DataFrame({
    "home_strength": home_strength,
    "away_strength": away_strength
})

team_strength["overall_strength"] = (
    team_strength["home_strength"] + team_strength["away_strength"]
) / 2

team_strength.head()

,home_strength,away_strength,overall_strength
76ers,0.552263,0.411184,0.481724
Bobcats,0.469248,0.245455,0.357351
Bucks,0.594527,0.397531,0.496029
Bulls,0.543118,0.388747,0.465932
Cavaliers,0.594507,0.407711,0.501109


# Stage 3: Prediction Logic
Turn team strengths into game predictions.

1. Build game-level dataset (recent form prep already included here)

In [238]:
home_games = df_modern[["hometeamName", "gameDate", "HOME_WIN"]].copy()
home_games.columns = ["team", "date", "win"]

away_games = df_modern[["awayteamName", "gameDate", "AWAY_WIN"]].copy()
away_games.columns = ["team", "date", "win"]

all_games = pd.concat([home_games, away_games])
all_games = all_games.sort_values(["team", "date"])

2. Recent form

In [239]:
all_games["recent_form"] = (
    all_games.groupby("team")["win"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean())
)

recent_form = all_games.groupby("team").tail(1)[["team", "recent_form"]]

3. Merge into team_strength

In [240]:
team_strength = team_strength.reset_index()

team_strength = team_strength.merge(
    recent_form,
    left_on="index",
    right_on="team",
    how="left"
)

4. Prediction function

In [241]:
def predict_game(home_team, away_team):

    home_row = team_strength[team_strength["index"] == home_team].iloc[0]
    away_row = team_strength[team_strength["index"] == away_team].iloc[0]

    home_score = (
        0.8 * home_row["overall_strength"] +
        0.2 * home_row["recent_form"]
    )

    away_score = (
        0.8 * away_row["overall_strength"] +
        0.2 * away_row["recent_form"]
    )

    total = home_score + away_score

    home_prob = home_score / total
    away_prob = away_score / total

    winner = home_team if home_prob > away_prob else away_team

    print(f"\n🏀 {home_team} vs {away_team}")
    print(f"{home_team}: {round(home_prob*100,1)}%")
    print(f"{away_team}: {round(away_prob*100,1)}%")
    print(f"Predicted Winner: {winner}")

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_prob": round(float(home_prob), 3),
        "away_prob": round(float(away_prob), 3),
        "predicted_winner": winner
    }

5. Test

In [242]:
predict_game("Heat", "Bucks")


🏀 Heat vs Bucks
Heat: 53.2%
Bucks: 46.8%
Predicted Winner: Heat


{'home_team': 'Heat',
 'away_team': 'Bucks',
 'home_prob': 0.532,
 'away_prob': 0.468,
 'predicted_winner': 'Heat'}